In [ ]:
# Cell 1: Config — auto-detect Colab VM vs local Mac
import os, subprocess

REPO   = "https://github.com/almaas-izdihar/ema-skd"
BRANCH = "experiment/confidence-filter"
DIR    = "/content/ema-skd"

ON_COLAB = os.path.exists('/content')

if ON_COLAB:
    # Pull latest from GitHub (results/ folder has logs pushed by training sessions)
    if not os.path.exists(DIR):
        subprocess.run(f"git clone {REPO} {DIR}", shell=True, check=True)
    subprocess.run(f"git -C {DIR} fetch origin {BRANCH}", shell=True, check=True)
    subprocess.run(f"git -C {DIR} checkout {BRANCH}", shell=True, check=True)
    subprocess.run(f"git -C {DIR} pull origin {BRANCH}", shell=True, check=True)
    os.chdir(DIR)
    BASELINE_LOG = f"{DIR}/results/baseline_log.txt"
    EMASKD_LOG   = f"{DIR}/results/emaskd_log.txt"
    OUT_DIR      = f"{DIR}/results"
else:
    # Local Mac — edit paths after colab download
    BASELINE_LOG = os.path.expanduser('~/box/projects/skripsi-kd/.claude/tmp/colab-output/baseline_log.txt')
    EMASKD_LOG   = os.path.expanduser('~/box/projects/skripsi-kd/.claude/tmp/colab-output/emaskd_log.txt')
    OUT_DIR      = os.path.dirname(BASELINE_LOG)

os.makedirs(OUT_DIR, exist_ok=True)

for p in [BASELINE_LOG, EMASKD_LOG]:
    status = '✓' if os.path.exists(p) else '✗ NOT FOUND'
    print(f'{status}  {p}')
print(f'Output dir: {OUT_DIR}')

In [ ]:
# Cell 2: Parse logs
import re
import pandas as pd

def parse_log(path):
    rows = []
    with open(path) as f:
        for line in f:
            if '[val]' not in line:
                continue
            def g(key):
                m = re.search(rf'\[{key} ([^\]]+)\]', line)
                return float(m.group(1)) if m else None
            ep = re.search(r'\[Epoch (\d+)\]', line)
            if not ep:
                continue
            rows.append({
                'epoch':    int(ep.group(1)),
                'top1':     g('val_top1_acc'),
                'top5':     g('val_top5_acc'),
                'val_loss': g('val_loss'),
                'ece':      g('ECE'),
                'aurc':     g('AURC'),
                'eaurc':    g('EAURC'),
            })
    return pd.DataFrame(rows).set_index('epoch')

def parse_train_log(path):
    rows = []
    with open(path) as f:
        for line in f:
            if '[train]' not in line:
                continue
            def g(key):
                m = re.search(rf'\[{key} ([^\]]+)\]', line)
                return float(m.group(1)) if m else None
            ep = re.search(r'\[Epoch (\d+)\]', line)
            if not ep:
                continue
            rows.append({
                'epoch':      int(ep.group(1)),
                'train_loss': g('train_loss'),
                'train_top1': g('train_top1_acc'),
                'gate_pct':   g('gate'),
                'tau':        g('tau'),
            })
    return pd.DataFrame(rows).set_index('epoch')

df_base       = parse_log(BASELINE_LOG)
df_ema        = parse_log(EMASKD_LOG)
df_base_train = parse_train_log(BASELINE_LOG)
df_ema_train  = parse_train_log(EMASKD_LOG)

print(f'Baseline epochs: {len(df_base)}  |  EMA-SKD epochs: {len(df_ema)}')

In [ ]:
# Cell 3: Summary table
last_base = df_base.iloc[-1]
last_ema  = df_ema.iloc[-1]

summary = pd.DataFrame({
    'Metric':   ['Top-1 Acc (%)', 'Top-5 Acc (%)', 'Val Loss', 'ECE (↓)', 'AURC (↓)', 'EAURC (↓)'],
    'Baseline': [last_base.top1, last_base.top5, last_base.val_loss, last_base.ece, last_base.aurc, last_base.eaurc],
    'EMA-SKD':  [last_ema.top1,  last_ema.top5,  last_ema.val_loss,  last_ema.ece,  last_ema.aurc,  last_ema.eaurc],
})
summary['Δ'] = summary['EMA-SKD'] - summary['Baseline']
summary['Δ%'] = summary.apply(
    lambda r: f"+{r['Δ']:.3f}" if r['Δ'] >= 0 else f"{r['Δ']:.3f}", axis=1
)
print(summary[['Metric', 'Baseline', 'EMA-SKD', 'Δ%']].to_string(index=False, float_format=lambda x: f'{x:.3f}'))
print()
print('Paper targets — Baseline: 75.55 ± 0.09  |  EMA-SKD: 79.19 ± 0.15')

# Crossover epoch
merged = df_ema[['top1']].rename(columns={'top1': 'ema'}).join(
    df_base[['top1']].rename(columns={'top1': 'base'}), how='inner')
cross = merged[merged['ema'] > merged['base']]
if cross.empty:
    print('\nEMA-SKD did not exceed baseline — check config or run more epochs')
else:
    ep = cross.index[0]
    print(f'\nEMA-SKD first exceeds baseline at epoch {ep}'
          f' (EMA {cross.loc[ep, "ema"]:.3f}% vs Base {cross.loc[ep, "base"]:.3f}%)')

In [ ]:
# Cell 4: Eval curves — Top-1, Val Loss, ECE, AURC
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('EMA-SKD (conf-gate) vs Baseline — CIFAR-100 / ResNet18', fontsize=13)

panels = [
    ('top1',     'Top-1 Accuracy (%)', False),
    ('val_loss', 'Val Loss',           True),
    ('ece',      'ECE (↓)',            True),
    ('aurc',     'AURC (↓)',           True),
]

for ax, (col, title, _) in zip(axes.flat, panels):
    ax.plot(df_base.index, df_base[col], label='Baseline', marker='o',
            markevery=10, linewidth=1.5, color='steelblue')
    ax.plot(df_ema.index,  df_ema[col],  label='EMA-SKD+gate', marker='s',
            markevery=10, linewidth=1.5, color='darkorange')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
out_path = os.path.join(OUT_DIR, 'eval_curves.png')
plt.savefig(out_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

In [ ]:
# Cell 5: Confidence gate analysis (EMA-SKD only)
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

if df_ema_train.empty or df_ema_train['gate_pct'].isna().all():
    print('No gate_pct data found — skipping')
else:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
    fig.suptitle('Confidence Gate Curriculum — EMA-SKD', fontsize=13)

    ax1.plot(df_ema_train.index, df_ema_train['gate_pct'],
             color='darkorange', linewidth=1.5)
    ax1.set_ylabel('Gate pass rate (%)')
    ax1.set_title('% samples entering EMA buffer per epoch')
    ax1.set_ylim(0, 105)
    ax1.grid(True, alpha=0.3)

    ax2.plot(df_ema_train.index, df_ema_train['tau'],
             color='steelblue', linewidth=1.5)
    ax2.set_ylabel('τ_t (threshold)')
    ax2.set_title('Curriculum threshold (strict→lenient)')
    ax2.set_xlabel('Epoch')
    ax2.set_ylim(0, 0.8)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    out_path = os.path.join(OUT_DIR, 'gate_curriculum.png')
    plt.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_path}')

    gp = df_ema_train['gate_pct'].dropna()
    print(f'\nGate pass rate — min: {gp.min():.1f}%  max: {gp.max():.1f}%  final: {gp.iloc[-1]:.1f}%')

In [ ]:
# Cell 6: Training loss comparison
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Training Dynamics — CIFAR-100 / ResNet18', fontsize=13)

ax1.plot(df_base_train.index, df_base_train['train_loss'],
         label='Baseline', color='steelblue', linewidth=1.5)
ax1.plot(df_ema_train.index,  df_ema_train['train_loss'],
         label='EMA-SKD+gate', color='darkorange', linewidth=1.5)
ax1.set_title('Train Loss')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(df_base_train.index, df_base_train['train_top1'],
         label='Baseline', color='steelblue', linewidth=1.5)
ax2.plot(df_ema_train.index,  df_ema_train['train_top1'],
         label='EMA-SKD+gate', color='darkorange', linewidth=1.5)
ax2.set_title('Train Top-1 Accuracy (%)')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
out_path = os.path.join(OUT_DIR, 'train_dynamics.png')
plt.savefig(out_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')